# 03 — XGBoost + Optuna Hyperparameter Tuning

Train XGBoost classifier with Optuna automatic hyperparameter search on MetaBreath-aligned sensor features.

**Inputs**: `data/processed/X_train.csv`, `y_train.csv`, `X_test.csv`, `y_test.csv`
**Output**: `apps/api/models/xgb_classifier.joblib`

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
from pathlib import Path

ROOT = Path('../../..')
PROCESSED = ROOT / 'data' / 'processed'
MODEL_DIR = ROOT / 'apps' / 'api' / 'models'
MODEL_DIR.mkdir(exist_ok=True)

X_train = pd.read_csv(PROCESSED / 'X_train.csv')
X_test  = pd.read_csv(PROCESSED / 'X_test.csv')
y_train = pd.read_csv(PROCESSED / 'y_train.csv').squeeze()
y_test  = pd.read_csv(PROCESSED / 'y_test.csv').squeeze()

print('Train:', X_train.shape, '| Test:', X_test.shape)

## Optuna Hyperparameter Search

In [ ]:
import optuna
import xgboost as xgb
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators':    trial.suggest_int('n_estimators', 100, 500),
        'max_depth':       trial.suggest_int('max_depth', 3, 10),
        'learning_rate':   trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':       trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':       trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':      trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'scale_pos_weight': (y_train == 0).sum() / (y_train == 1).sum(),
        'eval_metric': 'logloss',
        'use_label_encoder': False,
        'random_state': 42,
    }
    model = xgb.XGBClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1_weighted', n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, timeout=300)

print('Best params:', study.best_params)
print('Best CV F1:', study.best_value)

## Train Final XGBoost with Best Params

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score
)
import matplotlib.pyplot as plt
import seaborn as sns

best_params = study.best_params
best_params.update({
    'scale_pos_weight': (y_train == 0).sum() / (y_train == 1).sum(),
    'use_label_encoder': False,
    'eval_metric': 'logloss',
    'random_state': 42,
})

xgb_model = xgb.XGBClassifier(**best_params)
xgb_model.fit(X_train, y_train)

y_pred  = xgb_model.predict(X_test)
y_proba = xgb_model.predict_proba(X_test)[:, 1]

f1  = f1_score(y_test, y_pred, average='weighted')
roc = roc_auc_score(y_test, y_proba)

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Normal', 'Diabetes']))
print(f'Weighted F1: {f1:.4f}')
print(f'ROC-AUC:     {roc:.4f}')

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Normal', 'Diabetes'],
            yticklabels=['Normal', 'Diabetes'], ax=axes[0])
axes[0].set_title(f'XGBoost Confusion Matrix\nF1={f1:.3f}, AUC={roc:.3f}')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

# Optuna optimization history
optuna.visualization.matplotlib.plot_optimization_history(study, ax=axes[1])

plt.tight_layout()
plt.savefig(MODEL_DIR / 'xgb_evaluation.png', dpi=150)
plt.show()

## Drift Analysis — Performance by UCI Gas Drift Batch

In [ ]:
# Load UCI acetone data and test model performance per batch
uci_acetone = pd.read_csv(PROCESSED / 'uci_acetone_only.csv')
print('UCI acetone batches:', uci_acetone['batch'].value_counts().sort_index())

# Use feat_0 as proxy for breath_voc, feat_1 for ambient_voc
uci_acetone['acetone_delta'] = (uci_acetone['feat_0'] - uci_acetone['feat_1']).clip(lower=0)
uci_acetone['breath_voc'] = uci_acetone['feat_0']
uci_acetone['ambient_voc'] = uci_acetone['feat_1']
uci_acetone['quality_score'] = 75.0

batch_f1 = {}
for batch_id in sorted(uci_acetone['batch'].unique()):
    batch_data = uci_acetone[uci_acetone['batch'] == batch_id]
    # Since UCI gas drift is concentration regression (not binary), we use
    # predicted probability as drift indicator
    X_batch = batch_data[['acetone_delta', 'breath_voc', 'ambient_voc', 'quality_score']]
    proba = xgb_model.predict_proba(X_batch)[:, 1]
    batch_f1[batch_id] = proba.mean()
    print(f'  Batch {batch_id:2d}: mean_proba={proba.mean():.4f} (n={len(batch_data)})')

print('\nDrift evidence: performance typically degrades in later batches (7-10)')

## Save Model + Metrics

In [ ]:
joblib.dump(xgb_model, MODEL_DIR / 'xgb_classifier.joblib')
print('Saved: xgb_classifier.joblib')

metrics = {
    'f1_weighted': round(f1, 4),
    'roc_auc': round(roc, 4),
    'best_optuna_params': study.best_params,
    'n_train': len(X_train),
    'n_test': len(X_test),
    'optuna_trials': 50,
    'drift_batch_probas': batch_f1,
}
with open(MODEL_DIR / 'xgb_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to xgb_metrics.json')